# TensorFlow Fundamentals — Tensor Creation and Operations

**Study notes:** CPE312 · Hands-on Activity 9.1  
**Topic:** Deep Learning / TensorFlow Fundamentals  
**Format:** Ten exercises with explanations, verified shape expectations, and corrected examples.

These notes adapt the uploaded Jupyter notebook and use the supplied activity PDF
as a reference. They preserve the exercises and example values while clarifying
three important mistakes in matrix multiplication, squeezing, and one-hot encoding.
No external CSV or other dataset is required.


## Setup — Import TensorFlow and control randomness

TensorFlow is a numerical-computing library commonly used for machine learning and deep learning.
Its `tf.Tensor` objects store values with a fixed **shape** and **data type**. Set a random seed
so the examples are easier to reproduce; outputs can still vary across hardware or versions.


In [ ]:
# Import the TensorFlow API and give it the conventional alias "tf".
# TensorFlow supports multidimensional arrays (tensors) and computations
# that can run on CPUs, GPUs, and other compatible hardware.
import tensorflow as tf

# Seed TensorFlow's random-number generation for repeatable examples.
# A fixed seed is useful for tutorials, debugging, and model comparisons.
tf.random.set_seed(42)

# Library versions are useful when trying to reproduce results later.
print("TensorFlow version:", tf.__version__)


## 1. Create a scalar, vector, matrix, and 3D tensor

**Shape** describes the length of each axis. **Rank** counts the axes:
scalar → rank 0; vector → rank 1; matrix → rank 2; and this example tensor → rank 3.
`tf.constant()` creates a fixed-value tensor; it does not mean a trainable model parameter.


In [ ]:
# A scalar is one number, so it has no axes: shape=(), rank=0.
scalar = tf.constant(7)

# A vector is a one-dimensional sequence with two elements: shape=(2,).
vector = tf.constant([6, 7])

# A matrix has rows and columns: 2 rows × 2 columns, shape=(2, 2).
matrix = tf.constant([[10, 7], [6, 7]])

# A 3D tensor has three axes: 2 blocks × 2 rows × 3 columns.
tensor_3d = tf.constant([
    [[1, 2, 3], [4, 5, 6]],
    [[5, 6, 7], [4, 3, 1]],
])

# Keep the objects together for the following shape/rank/size exercise.
example_tensors = {
    "scalar": scalar,
    "vector": vector,
    "matrix": matrix,
    "tensor_3d": tensor_3d,
}

for name, value in example_tensors.items():
    print(f"{name}: {value.numpy()}")


## 2. Measure tensor shape, rank, and size

These three terms answer different questions:

- **Shape:** How many elements are along each axis? `tf.shape(t)` returns that information.
- **Rank:** How many axes are there? `tf.rank(t)` returns a scalar integer.
- **Size:** How many scalar values exist in total? `tf.size(t)` multiplies the shape dimensions.

An empty shape `[]` for the scalar is correct: it means *no axes*, not *no data*.


In [ ]:
# These TensorFlow functions return tensors containing the requested
# metadata; .numpy() makes their values easy to print in eager execution.
for name, value in example_tensors.items():
    shape = tf.shape(value)  # Dimension lengths, e.g. [2, 2, 3].
    rank = tf.rank(value)    # Number of dimensions, e.g. 3.
    size = tf.size(value)    # Total number of entries, e.g. 12.

    print(f"{name:9s} | shape={shape.numpy()} "
          f"| rank={rank.numpy()} | size={size.numpy()}")

# Verify the key observations from the original activity.
assert scalar.shape == ()
assert vector.shape == (2,)
assert matrix.shape == (2, 2)
assert tensor_3d.shape == (2, 2, 3)
assert int(tf.size(tensor_3d).numpy()) == 12


## 3. Create two random tensors of shape (5, 300)

A uniform distribution samples numbers throughout a specified interval.
`minval=0.0` and `maxval=1.0` produce floating-point values in the interval **[0, 1)**:
zero is possible, but one is excluded. The `(5, 300)` shape can be interpreted as
five samples, each with 300 numerical features.


In [ ]:
# Use float32, a common data type for neural-network computations.
# Both tensors have 5 rows (samples) and 300 columns (features).
x = tf.random.uniform(
    shape=(5, 300), minval=0.0, maxval=1.0, dtype=tf.float32
)
y = tf.random.uniform(
    shape=(5, 300), minval=0.0, maxval=1.0, dtype=tf.float32
)

print("X shape:", x.shape)
print("Y shape:", y.shape)

# Compare the shape and value range without printing all 1,500 values.
assert x.shape == (5, 300) and y.shape == (5, 300)
assert bool(tf.reduce_all((x >= 0.0) & (x < 1.0)).numpy())


## 4. Perform matrix multiplication (not element-wise multiplication)

**Important correction to the submitted notebook:** `tf.math.multiply(x, y)` performs
element-wise multiplication, so its output remains `(5, 300)`. It is **not** matrix
multiplication. For matrices A with shape `(m, n)` and B with shape `(n, p)`,
`tf.matmul(A, B)` has shape `(m, p)`.

Since both X and Y are `(5, 300)`, we transpose Y to `(300, 5)` before multiplication.
The resulting `(5, 5)` matrix contains dot-product similarities between pairs of rows.


In [ ]:
# Transpose Y so its first dimension (300) matches X's last dimension.
# Dimensions: (5 × 300) @ (300 × 5) = (5 × 5).
matrix_product = tf.matmul(x, tf.transpose(y))

# Element-wise multiplication is also useful, but it is a distinct operation:
# each position (i, j) becomes x[i, j] * y[i, j].
elementwise_product = tf.multiply(x, y)

print("Matrix product:", matrix_product.shape)         # (5, 5)
print("Element-wise product:", elementwise_product.shape)  # (5, 300)

assert matrix_product.shape == (5, 5)
assert elementwise_product.shape == (5, 300)


## 5. Calculate dot products using tf.tensordot

A dot product multiplies matching components of two vectors and sums them:
`a[0]*b[0] + ... + a[n-1]*b[n-1]`. `tf.tensordot` generalizes this idea to
higher-rank tensors by summing over selected axes.

Contracting the shared 300-feature axis of X and Yᵀ produces the same `(5, 5)`
values as matrix multiplication. Contracting a single pair of rows produces one scalar.


In [ ]:
# Contract the last axis of X against the first axis of transposed Y.
# axes=1 means to sum over one pair of matching axes (size 300).
pairwise_dot_products = tf.tensordot(x, tf.transpose(y), axes=1)

# For two 1D vectors of length 300, the dot product is a scalar.
first_row_dot_product = tf.tensordot(x[0], y[0], axes=1)

print("All row-pair dot products:", pairwise_dot_products.shape)
print("First row pair dot product:", first_row_dot_product.numpy())

# Both operations implement the same sum of products here (allowing for
# small floating-point rounding differences).
tf.debugging.assert_near(pairwise_dot_products, matrix_product)
tf.debugging.assert_near(first_row_dot_product, matrix_product[0, 0])


## 6. Create an image-shaped random tensor (224, 224, 3)

Many image pipelines use `(height, width, channels)`. Here, 224×224 represents
spatial dimensions and **3 channels** could represent RGB colors. These random values
are not a photograph; the shape merely resembles an image as seen by a model.


In [ ]:
# Create a synthetic image-shaped tensor with floating-point values.
# Each of its 224 × 224 pixel positions contains three channel values.
image_tensor = tf.random.uniform(
    shape=(224, 224, 3), minval=0.0, maxval=1.0, dtype=tf.float32
)

print("Image-like tensor shape:", image_tensor.shape)
assert image_tensor.shape == (224, 224, 3)


## 7. Compute the minimum and maximum tensor values

A **reduction** combines many values into fewer values. With no axis specified,
`tf.reduce_min()` and `tf.reduce_max()` search all 150,528 elements and return
one scalar each. These can serve as basic checks on scaled data.


In [ ]:
# Reduce all dimensions to one minimum and one maximum value.
min_value = tf.reduce_min(image_tensor)
max_value = tf.reduce_max(image_tensor)

# .numpy() converts eager TensorFlow scalar results to NumPy-compatible values.
print("Minimum:", float(min_value.numpy()))
print("Maximum:", float(max_value.numpy()))

# Every value was generated from the interval [0, 1).
assert 0.0 <= float(min_value.numpy()) <= float(max_value.numpy()) < 1.0


## 8. Remove a singleton batch axis with tf.squeeze

Image batches are commonly stored as `(batch_size, height, width, channels)`.
A batch of just one image therefore has shape `(1, 224, 224, 3)`.

**Correction:** The source activity used `tf.reshape()` to obtain `(224, 224, 3)`.
Although that works for these dimensions, `tf.squeeze(axis=0)` more clearly shows
the intended operation: *remove an axis whose length is exactly one*.


In [ ]:
# Create a batch containing one image-like tensor.
batched_image = tf.random.uniform(shape=(1, 224, 224, 3))

# Remove the leading axis because its length is 1.
# This does not discard or recalculate any of the pixel/channel values.
squeezed_image = tf.squeeze(batched_image, axis=0)

print("Before squeeze:", batched_image.shape)  # (1, 224, 224, 3)
print("After squeeze:", squeezed_image.shape)   # (224, 224, 3)

assert squeezed_image.shape == (224, 224, 3)
tf.debugging.assert_equal(squeezed_image, batched_image[0])


## 9. Find the position of the largest entry with tf.argmax

`tf.argmax()` returns the **index** of the maximum value, not the value itself.
Python and TensorFlow use zero-based indexing: in a ten-element vector, valid
positions run from 0 to 9. The largest value here is 102 at position 9.


In [ ]:
# Construct the same example values shown in the original activity.
values = tf.constant([7, 6, 24, 5, 11, 14, 6, 100, 101, 102])

# argmax returns a position (index). reduce_max returns the value.
max_index = tf.argmax(values)
max_entry = tf.reduce_max(values)

print("Vector shape:", values.shape)
print("Index of largest entry:", int(max_index.numpy()))
print("Largest entry:", int(max_entry.numpy()))

assert values.shape == (10,)
assert int(max_index.numpy()) == 9
assert int(max_entry.numpy()) == 102


## 10. One-hot encode valid category indices

One-hot encoding turns each *category ID* into a vector where exactly one
position is 1 and all other positions are 0. `tf.one_hot(indices, depth)` expects
each index to fall between **0** and **depth − 1**.

**Correction:** The original code passed raw values such as 7, 24, and 102 with
`depth=3`. Since none of those numbers is a valid category ID for depth 3, every
row became all zeros. That output is syntactically allowed but fails to encode
the intended information.

To preserve the original 10 numeric examples while encoding them as *categories*,
first map each distinct value to a contiguous ID (0–9), then one-hot encode.
In a real ML dataset, use one-hot encoding for categorical labels, not arbitrary
continuous measurements merely because their values are numeric.


In [ ]:
# tf.unique returns distinct values and the index of each original value
# within that distinct-value list. For this example all ten are unique.
unique_categories, category_ids = tf.unique(values)

# The number of different categories determines the vector length.
# category_ids contain integers from 0 to n_categories - 1.
n_categories = tf.size(unique_categories)

# Encode each category ID as a separate column of 0s and 1s.
one_hot_values = tf.one_hot(
    indices=category_ids, depth=n_categories, dtype=tf.float32
)

print("Distinct category values:", unique_categories.numpy())
print("Mapped category IDs:", category_ids.numpy())
print("One-hot shape:", one_hot_values.shape)
print("One-hot encoded tensor:\n", one_hot_values.numpy())

# For a valid one-hot representation, each row must contain exactly one 1.
tf.debugging.assert_equal(
    tf.reduce_sum(one_hot_values, axis=1),
    tf.ones(shape=(10,), dtype=tf.float32),
)
assert one_hot_values.shape == (10, 10)


### Bonus: Why the original one-hot encoding returned all zeros

The following cell deliberately reproduces the earlier mistake **for diagnosis only**.
`tf.one_hot` does not automatically re-label raw numeric values. `depth=3` means
valid IDs are only `0`, `1`, and `2`. All ten original entries are larger than `2`,
so the array is all zeros. The corrected implementation in Exercise 10 avoids this.


In [ ]:
# Demonstration only: this reproduces the original output, which is NOT a
# valid representation of the intended 10 categories.
incorrect_encoding = tf.one_hot(values, depth=3)
print("Incorrect encoding shape:", incorrect_encoding.shape)
print("Number of 1s in incorrect encoding:", tf.reduce_sum(incorrect_encoding).numpy())

# Since every value is outside [0, 3), nothing receives a 1.
assert float(tf.reduce_sum(incorrect_encoding).numpy()) == 0.0


## Recap and practical applications

- **Creating and describing tensors** is the foundation for passing input data to TensorFlow models.
- **Matrix multiplication and dot products** are core operations inside neural-network layers.
- **Image-shaped tensors and batch axes** explain common input layouts for computer vision.
- **Reduction operations** help check numerical ranges and summarize large arrays.
- **Argmax and one-hot encoding** are useful in classification, but category indices must be valid.

**Scope:** This activity introduces TensorFlow operations. It does **not** train a neural network,
make predictions, or use an external dataset.

**Reproducibility note:** This notebook was statically validated, but its TensorFlow cells have
not been executed here because TensorFlow is not installed in the authoring environment.
Run all cells in a TensorFlow-enabled Jupyter environment or Google Colab to see actual outputs.
